In [ ]:
%pip install repeat torch scikit-learn pandas onnx matplotlib tqdm

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import onnx

In [ ]:
design_cases = pd.read_csv("heat_transfer_data.csv", index_col=0)
design_cases.head()

In [ ]:
input_data = design_cases[["TEMPERATURE_1", "TEMPERATURE_2"]]
heat_flux = design_cases[["heat_flux_1", "heat_flux_2"]]

In [ ]:
axis = input_data.plot.scatter(x = 'TEMPERATURE_1', y = 'TEMPERATURE_2', c = 'tab:red')
axis.set_xlabel("TEMPERATURE_1 $[^{\circ}C]$")
axis.set_ylabel("TEMPERATURE_2 $[^{\circ}C]$")
fig = axis.get_figure()
fig.savefig("input_data.png")

In [ ]:
def get_torch_tensors(tables : pd.DataFrame):
    total = []
    for table in tables:
        tensors = {}
        for key, values in table.items():
            tensors[key] = torch.tensor(values.to_numpy(), dtype = torch.double).reshape(-1, 1)
        
        total.append(tensors)    
        
    return total

In [ ]:
input_data_train, input_data_test, heat_flux_train, heat_flux_test = train_test_split(
    input_data, heat_flux, test_size = 0.33, random_state = 42)

In [ ]:
summary = input_data_train.describe()
temperature_mean = torch.tensor(summary.loc["mean"].to_numpy(), dtype = torch.double)
temperature_std = torch.tensor(summary.loc["std"].to_numpy(), dtype = torch.double)

In [ ]:
input_data_train, input_data_test, heat_flux_train, heat_flux_test = get_torch_tensors((
    input_data_train, input_data_test, heat_flux_train, heat_flux_test
))

In [ ]:
class LinearModel(nn.Module):
    def __init__(self, temperature_mean, temperature_std):
        super().__init__()
        self.linear = nn.Linear(
            in_features = 2, 
            out_features = 1 
        )
        self.temperature_mean = temperature_mean
        self.temperature_std = temperature_std
        
    def forward(self, TEMPERATURE_1, TEMPERATURE_2):
        x = torch.cat([TEMPERATURE_1, TEMPERATURE_2], dim = 1)
        x = (x - self.temperature_mean) / self.temperature_std
        heat_flux_2 = self.linear(x)
        heat_flux_1 = - heat_flux_2
        return heat_flux_1, heat_flux_2

In [ ]:
torch.manual_seed(42)
model = LinearModel(temperature_mean, temperature_std).double()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr = 0.1)
num_epochs = 100
model.train()
for epoch in tqdm(range(num_epochs)):
    optimizer.zero_grad()
    outputs = model(**input_data_train)
    loss = criterion(outputs[1], heat_flux_train["heat_flux_2"])
    loss.backward()
    optimizer.step()

In [ ]:
model.eval()
with torch.no_grad():
    predictions = model(**input_data_test)
    mse = criterion(predictions[1], heat_flux_test["heat_flux_2"])
    print(f"Mean Squared Error (MSE): {mse.item():.4f}")

In [ ]:
torch.onnx.export(
    model,
    args = (
        torch.tensor([[5]], dtype = torch.double), 
        torch.tensor([[-15]], dtype = torch.double)
    ),
    f = "ml_heat_transfer.onnx",
    input_names = ["TEMPERATURE_1", "TEMPERATURE_2"],
    output_names = ["heat_flux_1", "heat_flux_2"],    
    )

In [ ]:
fig, ax = plt.subplots()
ax.plot(input_data_test["TEMPERATURE_1"], predictions[1], 'ro', markersize=5, label = "Predicted")
ax.plot(input_data_test["TEMPERATURE_1"], heat_flux_test["heat_flux_2"], 'b+', markersize=10, label = "Test")
ax.legend()